In [9]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from scikeras.wrappers import KerasClassifier
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import SGD, Adam

In [10]:
import sklearn
sklearn.__version__


'1.5.1'

In [2]:
#Step 2: Load Data
data = load_breast_cancer()
X = data.data
y = data.target

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [3]:
#Step 3: Model Function
def create_model(learning_rate=0.01,
                 num_layers=0,
                 num_neurons=64,
                 activation='relu',
                 optimizer='adam',
                 dropout_rate=0.0):

    model = keras.Sequential()

    # Input layer
    model.add(layers.Input(shape=(X.shape[1],)))

    # Hidden layers
    for _ in range(num_layers):
        model.add(layers.Dense(num_neurons, activation=activation))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    # Output layer
    model.add(layers.Dense(1, activation='sigmoid'))

    # Optimizer selection
    if optimizer == 'sgd':
        opt = SGD(learning_rate=learning_rate)
    elif optimizer == 'sgd_mom':
        opt = SGD(learning_rate=learning_rate, momentum=0.9)
    elif optimizer == 'adam':
        opt = Adam(learning_rate=learning_rate)

    # Compile
    model.compile(optimizer=opt,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model

In [4]:
#Step 4: Wrap Model
model = KerasClassifier(model=create_model, verbose=0)

In [5]:
#Step 5: Your Param Grid
param_grid = {
    'model__learning_rate': [0.1, 0.01, 0.001],
    'batch_size': [64, 256],
    'model__num_layers': [0, 1],
    'model__num_neurons': [64, 128],
    'model__activation': ['relu', 'tanh'],
    'model__optimizer': ['sgd', 'sgd_mom', 'adam'],
    'model__dropout_rate': [0.0, 0.3],
    'epochs': [8]
}

In [6]:
#Step 6: Grid Search
grid = GridSearchCV(estimator=model,
                    param_grid=param_grid,
                    cv=3,
                    n_jobs=-1)

grid_result = grid.fit(X, y)

C:\Users\user\anaconda3\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
C:\Users\user\anaconda3\Lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


In [7]:
#Step 7: Results
print("Best Accuracy:", grid_result.best_score_)
print("Best Parameters:\n", grid_result.best_params_)

Best Accuracy: 0.9806646245242737
Best Parameters:
 {'batch_size': 64, 'epochs': 8, 'model__activation': 'relu', 'model__dropout_rate': 0.3, 'model__learning_rate': 0.1, 'model__num_layers': 1, 'model__num_neurons': 64, 'model__optimizer': 'adam'}
